# Forecasting Future Coffee Prices


## Forecast the future 'Price Paid to Growers' for a specific coffee-producing country by using historical price data and other economic indicators from your dataset.

In [15]:
import pandas as pd

# Load the total production data
file_path = 'data/total-production.csv'
df_production = pd.read_csv(file_path)

# Display the first few rows of the dataframe
df_production.head()

,total_production,1990,1991,1992,1993,1994,1995,1996,1997,1998,...,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018
0,Angola,50.3450,79.3310,77.5200,32.6080,76.802,62.1090,70.925,64.330,85.3440,...,13.4200,34.9700,28.7150,32.7900,34.9350,39.4050,40.5150,44.8300,35.0060,40.3874
1,Bolivia (Plurinational State of),122.7770,103.5360,120.2350,50.8230,116.944,142.4850,124.579,140.719,137.9850,...,128.4751,117.2249,131.8354,105.2812,119.9122,99.8766,84.2191,77.9835,83.8112,82.5687
2,Brazil,27285.6286,27293.4934,34603.3542,28166.9786,28192.047,18060.2022,29196.743,26148.004,36760.8533,...,43976.8120,55428.4102,48591.8289,55418.0012,54688.9664,53304.7669,52870.5876,56788.1784,52739.8635,62924.8836
3,Burundi,487.3930,667.1990,620.2380,393.3540,664.143,433.9800,400.969,249.785,491.9920,...,111.6130,352.9776,204.1328,405.9615,163.2177,247.5500,274.1017,248.7933,202.1079,178.4206
4,Ecuador,1503.8150,2123.8240,1185.4800,2069.0070,2375.766,1888.2330,1992.914,1190.663,1205.9680,...,813.2849,853.9798,825.4144,828.1024,665.5450,644.0112,644.4926,644.8845,623.5744,601.0001


In [16]:
import os

# Path to the data directory
data_dir = 'data/'

# Dictionary to hold the dataframes
dfs = {}

# Loop through the files in the data directory
for filename in os.listdir(data_dir):
    if filename.endswith('.csv'):
        # Create a name for the dataframe from the filename
        df_name = filename.replace('.csv', '').replace('-', '_')
        
        # Construct the full file path
        file_path = os.path.join(data_dir, filename)
        
        # Load the csv file into a dataframe
        dfs[df_name] = pd.read_csv(file_path)

# Print the names of the loaded dataframes and their columns
for name, df in dfs.items():
    print(f"--- DataFrame: {name} ---")
    print(df.columns.tolist())
    print('\\n')

--- DataFrame: disappearance ---
['disappearance', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018']
\n
--- DataFrame: domestic_consumption ---
['domestic_consumption', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018']
\n
--- DataFrame: exports_calendar_year ---
['exports', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018']
\n
--- DataFrame: exports_crop_year ---
['exports_crop_year', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '19

In [17]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt 


### Step 1: Data Consolidation for Brazil

The goal of this step is to create a single, unified dataset for a major coffee-producing country. We will focus on **Brazil**, as it is the world's largest coffee producer and is well-represented in these files.

We will perform the following actions:
1.  **Isolate Brazil's Data**: For each relevant DataFrame (`total_production`, `domestic_consumption`, `exports_calendar_year`, and our target `prices_paid_to_growers`), we will filter out all data that doesn't pertain to Brazil.
2.  **Prepare for Merging**: We will select only the necessary columns (the year and the value) from each DataFrame and rename the value column to something descriptive (e.g., `Production`, `Consumption`). This prevents confusion when we combine them.
3.  **Merge DataFrames**: We will combine these individual, Brazil-specific DataFrames into one master DataFrame, using the `Year` column as the common link.

This will give us a clean, structured dataset ready for analysis and modeling.

In [ ]:
# --- Filter and Prepare Each DataFrame ---

# 1. Total Production
df_prod_brazil = dfs['total_production'][dfs['total_production']['total_production'] == 'Brazil'].copy()
df_prod_brazil = df_prod_brazil[['1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018']].T
df_prod_brazil.columns = ['Production']
df_prod_brazil.index.name = 'Year'

# 2. Domestic Consumption
df_cons_brazil = dfs['domestic_consumption'][dfs['domestic_consumption']['domestic_consumption'] == 'Brazil'].copy()
df_cons_brazil = df_cons_brazil[['1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018']].T
df_cons_brazil.columns = ['Consumption']
df_cons_brazil.index.name = 'Year'

# 3. Exports
df_exp_brazil = dfs['exports_calendar_year'][dfs['exports_calendar_year']['exports_calendar_year'] == 'Brazil'].copy()
df_exp_brazil = df_exp_brazil[['1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018']].T
df_exp_brazil.columns = ['Exports']
df_exp_brazil.index.name = 'Year'

# 4. Prices Paid to Growers (our target variable)
df_price_brazil = dfs['prices_paid_to_growers'][dfs['prices_paid_to_growers']['prices_paid_to_growers'] == 'Brazil'].copy()
df_price_brazil = df_price_brazil[['1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018']].T
df_price_brazil.columns = ['Price_Paid_to_Growers']
df_price_brazil.index.name = 'Year'


# --- Merge the DataFrames ---
df_brazil = pd.concat([df_prod_brazil, df_cons_brazil, df_exp_brazil, df_price_brazil], axis=1)

# Convert index to integer for clean plotting
df_brazil.index = df_brazil.index.astype(int)

# Display the first few rows of the new merged dataframe
df_brazil.head()

1. Data Merging and Feature Engineering:

2. Exploratory Data Analysis (EDA):


3. Time-Series Modeling (Regression):